# 1. Introduction
The goal is to create a RAG application using a simple LLM in a agentic setting so that you do not waste precious tokens from high end models for simple rag

In [2]:
print("Notebook ready")

Notebook ready


In [3]:
# install libraries
%pip install pymupdf
import fitz


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 768.5/768.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.9/556.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 640.4/640.4 kB 9.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

data_dir = Path("data")
pdf_files = list(data_dir.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF(s): {[f.name for f in pdf_files]}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

all_chunks = []

for pdf_path in pdf_files:
    doc = fitz.open(pdf_path)
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()
        if not text.strip():
            continue  # Skip blank pages
        
        page_splits = splitter.split_text(text)
        for split_text in page_splits:
            all_chunks.append({
                "text": split_text,
                "metadata": {
                    "source": pdf_path.name,
                    "page": page_num
                }
            })

print(f"Total chunks created across all documents: {len(all_chunks)}")
if all_chunks:
    print("\nSample chunk:")
    print(all_chunks[0])

Found 1 PDF(s): ['Grant Agreement - GAP-101182461_local_copy.pdf']
Total chunks created across all documents: 916

Sample chunk:
{'text': 'Project: 101182461 — EmergeNOW — HORIZON-CL6-2024-FARM2FORK-02\nHE MGA — Multi & Mono: v1.2\nEUROPEAN RESEARCH EXECUTIVE AGENCY (REA)\n \n \nREA.B – Green Europe\nB.2 – Farm to fork, Communities Development and Climate Action\nGRANT AGREEMENT\nProject 101182461  —  EmergeNOW\nPREAMBLE\nThis Agreement (‘the Agreement’) is between the following parties:\non the one part,\nthe European Research Executive Agency (REA) (‘EU executive agency’ or ‘granting authority’),\nunder the powers delegated by the European Commission (‘European Commission’),\nand\non the other part,\n1. ‘the coordinator’:\nETHNIKO KENTRO EREVNAS KAI TECHNOLOGIKIS ANAPTYXIS (CERTH), PIC\n998802502, established in CHARILAOU THERMI ROAD 6 KM, THERMI THESSALONIKI\n57001, Greece,', 'metadata': {'source': 'Grant Agreement - GAP-101182461_local_copy.pdf', 'page': 1}}


In [6]:
%pip install sentence-transformers chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 11.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 11.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 11.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 11.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 11.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 11.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 11.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 11.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 11.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12

In [7]:
import chromadb
from chromadb.utils import embedding_functions

# 1. Initialize persistent local DB (saves to a local folder named "rag_storage")
client = chromadb.PersistentClient(path="./rag_storage")

# 2. Pick the embedding function (BAAI/bge-base-en-v1.5 offers high accuracy with low memory)
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-base-en-v1.5"
)

# 3. Create or access the collection
collection = client.get_or_create_collection(
    name="grant_documents",
    embedding_function=emb_fn
)

# 4. Format chunks for ChromaDB insertion
texts = [c["text"] for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]
ids = [f"chunk_{i}" for i in range(len(all_chunks))]

# 5. Add to vector store (this embeds and indexes automatically)
collection.add(
    documents=texts,
    metadatas=metadatas,
    ids=ids
)

print(f"Indexed {collection.count()} chunks into ChromaDB.")

/Users/fanis/Desktop/Projects/Personal/Agentic_AI_playground/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8791.70it/s]


Indexed 916 chunks into ChromaDB.


In [8]:
query = "What are the project deliverables or reporting deadlines?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

for i, doc in enumerate(results["documents"][0]):
    meta = results["metadatas"][0][i]
    print(f"--- Result {i+1} [Source: {meta['source']}, Page {meta['page']}] ---")
    print(doc[:300] + "...\n")

--- Result 1 [Source: Grant Agreement - GAP-101182461_local_copy.pdf, Page 39] ---
milestones, outputs/outcomes, critical risks, indicators, etc; if any), in the Portal Continuous
Reporting tool and in accordance with the timing and conditions it sets out (as agreed with the granting
authority).
Standardised deliverables (e.g. progress reports not linked to payments, reports on cu...

--- Result 2 [Source: Grant Agreement - GAP-101182461_local_copy.pdf, Page 91] ---
Project: 101182461 — EmergeNOW — HORIZON-CL6-2024-FARM2FORK-02
LIST OF DELIVERABLES
Deliverables
Grant Preparation (Deliverables screen) — Enter the info.
The labels used mean:
Public — fully open (
 automatically posted online)
Sensitive — limited under the conditions of the Grant Agreement
EU clas...

--- Result 3 [Source: Grant Agreement - GAP-101182461_local_copy.pdf, Page 11] ---
costs, financial support to third parties and exempted specific cost categories, if any)
-
VAT: Yes
-
Other ineligible costs
Budget flexibili